# VAE — Auto-Encoding Variational Bayes

**Статья:** Kingma & Welling — *Auto-Encoding Variational Bayes* (arXiv:1312.6114, ICLR 2014).

🌳 **Узел дерева:** Generative → latent-variable → амортизированный вариационный вывод.

**Что строим и обучаем:** энкодер-декодер на реальных 8×8 цифрах (sklearn `load_digits`), с нуля обучаем два варианта — VAE и обычный AE той же ёмкости — и меряем, кто из них после обучения умеет *генерировать*: сэмплируем $z\sim N(0,I)$, декодируем, смотрим, попадает ли картинка на многообразие реальных цифр.

**Компьют:** CPU, пара минут. Данные маленькие и реальные, GPU не нужен.

**Parts:**
1. Reparameterization trick — дифференцируемый сэмплинг (стержневая идея статьи).
2. Closed-form KL к $N(0,I)$ — регуляризатор ELBO.
3. Reconstruction-терм (Bernoulli NLL).
4. `forward` + сборка ELBO-лосса и обучение обеих веток.
5. Генеративный тест: сэмпл из приора → многообразие, VAE vs AE (эффект статьи).

Метрика эффекта — среднее 1-NN расстояние от декодированного приор-сэмпла до реального train-множества (ниже = на многообразии = когерентная цифра).

## ⛏ Эксперименты-первыми (заполни ДО чтения метода)

Открой Results/Experiments статьи (или прикинь по памяти) и запиши три пункта. Потом сверимся.

> **(a) Что результаты реально показали сверх abstract?** (какая задача, какие датасеты, что именно улучшилось vs предыдущие оценщики нижней границы)
>
> …

> **(b) Самый слабый эксперимент / чем бы фальсифицировала?**
>
> …

> **(c) Где метод буксует / чего НЕ протестировали?** (размерность латента, тип декодера, что с непрерывными данными)
>
> …

## Зачем это

Хотим обучаемую генеративную модель со скрытой переменной: $p(x)=\int p(x\mid z)\,p(z)\,dz$, приор $p(z)=N(0,I)$, декодер $p_\theta(x\mid z)$ — нейросеть. Прямая максимизация $\log p(x)$ требует неберущегося интеграла по $z$; апостериор $p(z\mid x)$ тоже неберущийся. Классический вариационный вывод вводит приближение $q_\phi(z\mid x)$ и максимизирует нижнюю границу (ELBO), но упирается в два места: (1) для каждого $x$ надо решать отдельную оптимизацию по вариационным параметрам; (2) градиент по $\phi$ от $E_{q_\phi}[\cdot]$ имеет высокую дисперсию.

**Фишка статьи — два хода, которые снимают оба затыка:**

- **Амортизация:** $q_\phi(z\mid x)=N(\mu_\phi(x),\operatorname{diag}\sigma_\phi^2(x))$ — одна сеть-энкодер выдаёт параметры апостериора для любого $x$. Не оптимизация на каждую точку, а один forward.
- **Reparameterization trick:** вместо того чтобы семплить $z\sim q_\phi$ (недифференцируемо по $\phi$), пишем $z=\mu+\sigma\odot\varepsilon$, $\varepsilon\sim N(0,I)$. Стохастику вынесли в $\varepsilon$, не зависящую от параметров, — и градиент течёт в $\mu,\sigma$ обычным backprop, с низкой дисперсией. Это и есть то, что делает всю конструкцию обучаемой end-to-end.

**Почему такой датасет:** нужен *реальный* набор картинок, на котором «попал/не попал на многообразие» осмысленно и при этом CPU-friendly. `load_digits` — 1797 настоящих рукописных 8×8 цифр; маленький и честный, MNIST-подобный.

## Метод в одной фразе (Фейнман)

**Задача.** Обучить декодер $p_\theta(x\mid z)$ так, чтобы, потянув $z$ из фиксированного приора $N(0,I)$, получать реальные объекты.

**Механизм.** Максимизируем ELBO — нижнюю границу лог-правдоподобия:
$$\log p_\theta(x)\ \ge\ \underbrace{E_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]}_{\text{reconstruction}}\ -\ \underbrace{\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big)}_{\text{регуляризатор}}.$$
Первый терм тянет декодер восстанавливать $x$ из его кода. Второй **прижимает** апостериор каждого объекта к приору $N(0,I)$ — за счёт этого агрегированное распределение кодов совпадает с тем, из чего мы потом семплим на генерации. Нет KL → у кодов нет причин совпадать с $N(0,I)$ → семпл из приора декодируется в мусор.

**Математика двух дыр.**
- KL двух гауссиан берётся в замкнутом виде (App. B статьи): это функция только от $\mu_j$ и $\log\sigma_j^2$, суммируется по латентным осям.
- Мат-ожидание в reconstruction оцениваем одним семплом $z$, полученным через reparameterization; для бинаризованных пикселей $\log p_\theta(x\mid z)$ — это Bernoulli log-likelihood (то есть $-\mathrm{BCE}$).

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** Почему нельзя просто взять $z=\mu$ (среднее апостериора) и не семплить вовсе? Что тогда перестаёт работать в обучении декодера как *генеративной* модели, а не как автоэнкодера?

**Ваш ответ:** …

In [ ]:
# --- setup (готово, просто запусти) ---
# pip install numpy torch scikit-learn matplotlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cpu")

# Данные: реальные 8x8 цифры, пиксели 0..16 -> [0,1].
digits = load_digits()
X = digits.data.astype(np.float32) / 16.0          # [N,64] в [0,1]
y = digits.target.astype(np.int64)                 # [N]
N, D = X.shape                                      # 1797, 64
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
Xtr_t = torch.from_numpy(Xtr).to(device)
Xte_t = torch.from_numpy(Xte).to(device)

Z_DIM = 16; H = 128; EPOCHS = 60; BATCH = 128; LR = 1e-3
K = 500   # число приор-сэмплов для генеративного теста
print(f"data: {N} real 8x8 digits (train {Xtr.shape[0]} / test {Xte.shape[0]}), z_dim={Z_DIM}")

## Part 1 — Reparameterization trick

Стержневая дыра. Энкодер выдаёт $\mu$ и $\log\sigma^2$ (`logvar` — логарифм дисперсии, чтобы $\sigma>0$ без ограничений). Нужен **дифференцируемый** семпл из $q(z\mid x)=N(\mu,\operatorname{diag}\sigma^2)$: такой, чтобы `.backward()` доводил градиент до `mu` и `logvar`.

Реализуй `reparameterize(mu, logvar)`.

In [ ]:
def reparameterize(mu, logvar):
    """Дифференцируемый семпл z из q(z|x) = N(mu, diag(sigma^2)), где sigma^2 = exp(logvar).

    Стохастичность должна идти от шума, НЕ зависящего от (mu, logvar), чтобы
    градиент от z тёк в оба аргумента обычным backprop.

    Args:
        mu:     [B, z] float — среднее апостериора.
        logvar: [B, z] float — log дисперсии апостериора (то есть log(sigma^2)).
    Returns:
        z: [B, z] float — один семпл из N(mu, diag(sigma^2)), дифференцируемый по mu и logvar.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# (1) grad-flow: градиент от z доходит и до mu, и до logvar
torch.manual_seed(0)
mu = torch.randn(64, Z_DIM, requires_grad=True)
logvar = torch.randn(64, Z_DIM, requires_grad=True)
z = reparameterize(mu, logvar)
assert z.shape == mu.shape, f"форма z {tuple(z.shape)} != mu {tuple(mu.shape)}"
z.sum().backward()
assert mu.grad is not None and mu.grad.abs().sum() > 0, "нет градиента в mu — семпл не дифференцируем по среднему"
assert logvar.grad is not None and logvar.grad.abs().sum() > 0, "нет градиента в logvar — шум не масштабируется через sigma(logvar)"

# (2) статистика: при большом B среднее семпла ~ mu, std ~ exp(0.5*logvar)
with torch.no_grad():
    mu0 = torch.full((200000, 4), 2.0)
    lv0 = torch.log(torch.full((200000, 4), 0.25))   # sigma = 0.5
    zz = reparameterize(mu0, lv0)
    emp_mean = zz.mean(0).mean().item()
    emp_std  = zz.std(0).mean().item()
print(f"Expected \u2248 mean 2.000, std 0.500 | у тебя mean {emp_mean:.3f}, std {emp_std:.3f}")
assert abs(emp_mean - 2.0) < 0.02, "среднее семпла должно быть ~mu"
assert abs(emp_std - 0.5) < 0.02, "разброс семпла должен быть ~exp(0.5*logvar); проверь связь logvar->sigma"
print("ok")

❓ **Inline Question 2.** Если бы энкодер выдавал напрямую $\sigma$ (а не $\log\sigma^2$), что могло бы сломаться в обучении, и почему параметризация через `logvar` удобнее?

**Ваш ответ:** …

## Part 2 — Closed-form KL к $N(0,I)$

Второй терм ELBO — $\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big)$ с $p(z)=N(0,I)$. Для диагональной гауссианы он берётся в замкнутом виде (App. B статьи) — как функция $\mu$ и `logvar`, без семплинга.

Реализуй `kl_standard_normal(mu, logvar)`: посчитай KL для каждого объекта (сумма по латентным осям), затем усредни по батчу — так он войдёт в лосс на равных с reconstruction-термом.

In [ ]:
def kl_standard_normal(mu, logvar):
    """KL( N(mu, diag(exp(logvar))) || N(0, I) ), в замкнутой форме (paper App. B).

    Считается аналитически из (mu, logvar), без семплинга. Значение неотрицательно
    и равно нулю ровно при mu=0, logvar=0 (апостериор совпал с приором).

    Args:
        mu:     [B, z] float — среднее апостериора.
        logvar: [B, z] float — log дисперсии апостериора.
    Returns:
        скаляр float — KL, просуммированный по латентным осям для каждого объекта
                       и усреднённый по батчу.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# (1) KL = 0 ровно на приоре (mu=0, logvar=0)
z0 = torch.zeros(8, Z_DIM)
k0 = kl_standard_normal(z0, z0)
print(f"Expected \u2248 0.000 на приоре | у тебя {float(k0):.4f}")
assert abs(float(k0)) < 1e-6, "на mu=0,logvar=0 KL должен быть ровно 0"

# (2) неотрицательность на случайном апостериоре
torch.manual_seed(1)
mu = torch.randn(256, Z_DIM); logvar = torch.randn(256, Z_DIM)
k = kl_standard_normal(mu, logvar)
assert float(k) > 0, "KL неотрицателен и >0 для апостериора != приора"

# (3) сверка с НЕЗАВИСИМОЙ Monte-Carlo оценкой KL на скалярном примере:
#     q = N(1, e) (mu=1, logvar=1) против p = N(0,1). MC-оценка E_q[log q - log p].
torch.manual_seed(7)
mu1 = torch.tensor([[1.0]]); lv1 = torch.tensor([[1.0]])
with torch.no_grad():
    s = torch.exp(0.5 * lv1)
    zq = mu1 + s * torch.randn(4000000, 1)
    logq = -0.5 * (((zq - mu1) / s) ** 2) - torch.log(s) - 0.5 * np.log(2 * np.pi)
    logp = -0.5 * (zq ** 2) - 0.5 * np.log(2 * np.pi)
    kl_mc = float((logq - logp).mean())
got = float(kl_standard_normal(mu1, lv1))
print(f"Expected \u2248 {kl_mc:.4f} (MC-оценка) | closed-form у тебя {got:.4f}")
assert abs(got - kl_mc) < 5e-3, "закрытая форма расходится с MC-оценкой KL; проверь знак и суммирование по осям"
print("ok")

## Part 3 — Reconstruction-терм (Bernoulli NLL)

Первый терм ELBO — $E_{q}[\log p_\theta(x\mid z)]$. Декодер выдаёт **логиты** по пикселям; пиксели в $[0,1]$ трактуем как бернуллиевы вероятности (конвенция `pytorch/examples/vae`). Значит $-\log p_\theta(x\mid z)$ — это BCE-with-logits, просуммированный по пикселям одного объекта и усреднённый по батчу (чтобы шкала совпадала с KL).

Реализуй `recon_bce(x_logits, x)`.

In [ ]:
def recon_bce(x_logits, x):
    """Bernoulli negative log-likelihood реконструкции (BCE по логитам).

    Пиксели x в [0,1] — бернуллиевы вероятности; x_logits — логиты декодера.
    Суммируется по пикселям каждого объекта и усредняется по батчу (та же шкала, что у KL).

    Args:
        x_logits: [B, D] float — логиты декодера (до сигмоиды).
        x:        [B, D] float — целевые пиксели в [0,1].
    Returns:
        скаляр float — суммарный-по-пикселям, средний-по-батчу BCE.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# сверка с НЕЗАВИСИМЫМ эталоном: Bernoulli NLL из вероятностей (sigmoid + ручной log-prob),
# просуммированный по пикселям и усреднённый по батчу.
torch.manual_seed(2)
xl = torch.randn(32, D); xt = torch.rand(32, D)
with torch.no_grad():
    p = torch.sigmoid(xl).clamp(1e-7, 1 - 1e-7)
    nll = -(xt * torch.log(p) + (1 - xt) * torch.log(1 - p))   # per-pixel Bernoulli NLL
    ref = float(nll.sum() / xl.shape[0])                        # sum по пикселям / batch
got = float(recon_bce(xl, xt))
print(f"Expected \u2248 {ref:.4f} | у тебя {got:.4f}")
assert abs(got - ref) < 1e-3, "не совпало с Bernoulli NLL (sum по пикселям / batch)"
# reduction должен быть sum-по-пикселям, а не mean: масштаб ~ десятки на 64 пикселя
assert got > 1.0, "похоже на mean, а не sum по пикселям — KL тогда задавит reconstruction"
print("ok")

❓ **Inline Question 3.** Почему reconstruction суммируют по пикселям, а KL суммируют по латентным осям — и оба потом усредняют по батчу? Что было бы, если reconstruction усреднить по пикселям (`reduction='mean'`)? Как это сдвинет баланс двух термов?

**Ваш ответ:** …

## Part 4 — `forward`, сборка ELBO и обучение

Модель — одна, с флагом `variational`, который включает/выключает вклад статьи (это и есть наша ablation-ось):
- `variational=True` → VAE: энкодер даёт $(\mu,\log\sigma^2)$, латент семплится через `reparameterize`, в лосс входит KL.
- `variational=False` → обычный AE: код детерминированный, KL нет.

Архитектура и ёмкость в обеих ветках одинаковы — единственная разница в KL + семплинге.

Две дыры:
1. `Autoencoder.forward` — как латент получается из $(\mu,\log\sigma^2)$ в каждом режиме.
2. В `train` — сборка обучающего лосса из твоих термов.

In [ ]:
# --- модель (энкодер/декодер даны; forward — дыра ниже) ---
class Autoencoder(nn.Module):
    def __init__(self, variational: bool):
        super().__init__()
        self.variational = variational
        self.enc = nn.Sequential(nn.Linear(D, H), nn.ReLU())
        self.fc_mu = nn.Linear(H, Z_DIM)
        self.fc_logvar = nn.Linear(H, Z_DIM)   # используется только при variational
        self.dec = nn.Sequential(nn.Linear(Z_DIM, H), nn.ReLU(), nn.Linear(H, D))

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        return self.dec(z)                      # логиты

    forward = None  # определяется в следующей ячейке-задании

In [ ]:
def forward(self, x):
    """Прямой проход: закодировать x, получить латент, декодировать в логиты.

    В варианте VAE (self.variational=True) латент — стохастический семпл из апостериора
    через reparameterize; в варианте AE (False) — детерминированный код (среднее апостериора,
    без семплинга). fc_logvar в AE-ветке в латент не влияет.

    Args:
        x: [B, D] float — вход в [0,1].
    Returns:
        (x_logits, mu, logvar):
            x_logits: [B, D] float — логиты реконструкции,
            mu:       [B, z] float,
            logvar:   [B, z] float.
    """
    # YOUR CODE HERE
    raise NotImplementedError

Autoencoder.forward = forward

In [ ]:
def train(variational: bool, seed: int = SEED):
    """Обучить одну ветку и вернуть модель.

    Лосс на батч — оценка -ELBO: reconstruction всегда; KL-терм (regularizer) добавляется
    ТОЛЬКО в variational-ветке. В AE-ветке минимизируется чистая реконструкция.
    Оптимизатор и цикл даны — впиши только сборку `loss` из recon_bce / kl_standard_normal.

    Args:
        variational: bool — VAE (True) или AE (False).
        seed: int.
    Returns:
        обученная модель Autoencoder.
    """
    torch.manual_seed(seed)
    model = Autoencoder(variational=variational).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    n = Xtr_t.shape[0]
    for _ in range(EPOCHS):
        perm = torch.randperm(n)
        for i in range(0, n, BATCH):
            xb = Xtr_t[perm[i:i + BATCH]]
            x_logits, mu, logvar = model(xb)
            # YOUR CODE HERE: собери loss (оценку -ELBO) из recon_bce и, если variational, kl_standard_normal
            raise NotImplementedError
            opt.zero_grad(); loss.backward(); opt.step()
    return model

In [ ]:
# --- обучаем обе ветки (пара минут на CPU) ---
print("training VAE ...");     vae = train(variational=True)
print("training plain AE ..."); ae  = train(variational=False)
print("done")

In [ ]:
# sanity: обе ветки реально реконструируют тест (forward + лосс собраны верно)
@torch.no_grad()
def reconstruct_bce(model):
    model.eval(); x_logits, _, _ = model(Xte_t)
    return recon_bce(x_logits, Xte_t).item()
bce_vae, bce_ae = reconstruct_bce(vae), reconstruct_bce(ae)
print(f"Expected \u2248 VAE ~23, AE ~16 (обе < 60) | у тебя VAE {bce_vae:.2f}, AE {bce_ae:.2f}")
assert bce_vae < 60.0, f"VAE не реконструирует (BCE {bce_vae:.1f}) — проверь forward/лосс"
assert bce_ae  < 60.0, f"AE не реконструирует (BCE {bce_ae:.1f})"
print("ok")

## Part 5 — Генеративный тест: эффект статьи

Теперь главное. Обученный декодер — генеративная модель, только если, потянув $z\sim N(0,I)$ (приор) и декодировав, мы попадаем на многообразие реальных цифр. Мерим это средним 1-NN расстоянием от каждого сгенерированного изображения до реального train-множества (ниже = на многообразии).

Тривиальные «генераторы» как нижний пол той же метрикой: всё-чёрное и картинка-среднее (размытая усреднённая цифра — обманчиво сильный пол). Настоящая генеративная модель должна лечь ближе к многообразию, чем они.

Вспомогательный код (семплинг из приора, метрика, коды) дан. Запусти и посмотри на контраст VAE vs AE.

In [ ]:
@torch.no_grad()
def sample_prior(model, k, seed=123):
    """Декодировать k латентов z~N(0,I). Возвращает probs [k,64] в [0,1]."""
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(k, Z_DIM, generator=g)
    return torch.sigmoid(model.decode(z)).cpu().numpy()

@torch.no_grad()
def code_stats(model):
    """Агрегированный апостериор на тесте: близость кодов к N(0,I). (mean|mu|, mean|std-1|)."""
    model.eval(); mu, _ = model.encode(Xte_t)
    return mu.mean(0).abs().mean().item(), (mu.std(0) - 1.0).abs().mean().item()

nn1 = NearestNeighbors(n_neighbors=1).fit(Xtr)   # многообразие реальных цифр
def manifold_dist(imgs):
    """Среднее 1-NN евклидово расстояние до реального train-множества."""
    return float(nn1.kneighbors(imgs)[0].mean())

vae_prior = sample_prior(vae, K); ae_prior = sample_prior(ae, K)
zeros_img = np.zeros((K, D), dtype=np.float32)
mean_img  = np.tile(Xtr.mean(0, keepdims=True), (K, 1))

d_real  = manifold_dist(Xte)
d_vae   = manifold_dist(vae_prior)
d_ae    = manifold_dist(ae_prior)
d_zeros = manifold_dist(zeros_img)
d_mean  = manifold_dist(mean_img)
vae_m, vae_s = code_stats(vae); ae_m, ae_s = code_stats(ae)

print(f"code-space |mean|   VAE={vae_m:.3f}  AE={ae_m:.3f}  (VAE ~0 = prior-centered)")
print("GENERATIVE TEST — 1-NN dist to real digits (lower=better):")
print(f"  real test (floor) : {d_real:.3f}")
print(f"  VAE  prior samples: {d_vae:.3f}   <- on-manifold")
print(f"  AE   prior samples: {d_ae:.3f}   <- off-manifold")
print(f"  trivial data-mean : {d_mean:.3f}")
print(f"  trivial all-zeros : {d_zeros:.3f}")

### Ключевая фигура — код-спейс и генеративный тест

Слева: агрегированный апостериор. VAE прижат к $N(0,I)$ (среднее ≈ 0), AE — гуляет. Справа: 1-NN расстояние до многообразия для каждого «генератора».

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar([0, 1], [vae_m, ae_m]); ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['VAE', 'AE'])
ax[0].set_ylabel('code-space |mean|'); ax[0].set_title('aggregated posterior vs N(0,I)')
ax[0].axhline(0, color='k', lw=0.5)
names = ['real\n(floor)', 'VAE', 'AE', 'data-mean', 'all-zeros']
vals  = [d_real, d_vae, d_ae, d_mean, d_zeros]
cols  = ['tab:green', 'tab:blue', 'tab:red', 'tab:gray', 'tab:gray']
ax[1].bar(range(5), vals, color=cols); ax[1].set_xticks(range(5)); ax[1].set_xticklabels(names)
ax[1].axhline(d_mean, color='tab:gray', ls='--', lw=1)
ax[1].set_ylabel('1-NN dist to manifold'); ax[1].set_title('prior-sample -> real digits (lower=better)')
plt.tight_layout(); plt.show()
# посмотрим и глазами: 8 приор-сэмплов VAE vs AE
fig, ax = plt.subplots(2, 8, figsize=(11, 3))
for j in range(8):
    ax[0, j].imshow(vae_prior[j].reshape(8, 8), cmap='gray_r'); ax[0, j].axis('off')
    ax[1, j].imshow(ae_prior[j].reshape(8, 8), cmap='gray_r'); ax[1, j].axis('off')
ax[0, 0].set_title('VAE', loc='left'); ax[1, 0].set_title('AE', loc='left'); plt.show()

In [ ]:
# ЭФФЕКТ СТАТЬИ (контролируемая ablation): единственная разница между ветками — KL + reparam.
# Пороги НЕ подгонялись под прогон; ниже — наблюдённые запасы над assert-границами.

# code-space: апостериор VAE центрирован на приоре, AE — нет
print(f"VAE |mean|={vae_m:.3f} (<0.20), AE |mean|={ae_m:.3f} (> VAE+1.0)")
assert vae_m < 0.20, f"VAE код не центрирован на приоре (|mean|={vae_m:.3f})"          # наблюдено 0.013
assert ae_m > vae_m + 1.0, f"AE неожиданно центрирован (|mean|={ae_m:.3f})"            # наблюдено 2.457 > 1.013

# VAE-сэмплы из приора ближе к многообразию, чем ОБА тривиальных пола
assert d_vae < d_mean,  f"VAE хуже пола data-mean ({d_vae:.3f} vs {d_mean:.3f})"       # 1.326 < 1.510
assert d_vae < d_zeros, f"VAE хуже пола all-zeros ({d_vae:.3f} vs {d_zeros:.3f})"      # 1.326 < 3.174

# главное: с KL+reparam приор-сэмплы НА многообразии, без них — ОФФ; и AE не бьёт даже data-mean
print(f"d_vae={d_vae:.3f}  d_ae={d_ae:.3f}  gap={d_ae-d_vae:+.3f} (>0.30)  d_mean={d_mean:.3f}")
assert d_ae > d_vae + 0.30, f"ЭФФЕКТ НЕ ВОСПРОИЗВЁЛСЯ: VAE {d_vae:.3f} не явно ниже AE {d_ae:.3f}"  # gap 0.664
assert d_ae > d_mean, f"ЭФФЕКТ СЛАБ: AE ({d_ae:.3f}) должен быть хуже пола data-mean ({d_mean:.3f})"  # 1.990 > 1.510
print("ok — эффект статьи воспроизведён: VAE на многообразии, AE — нет")

## Фишка — где она всплыла

Обе ветки одинаково хорошо *реконструируют* (BCE VAE≈23, AE≈16 — AE даже чуть лучше, KL на него не давит). Разница появляется ровно в тот момент, когда мы перестаём кормить декодер кодами *реальных* объектов и тянем $z$ из приора. VAE туда клал коды не абы как — KL прижимал агрегированный апостериор к $N(0,I)$, поэтому приор и то, чем декодер кормили при обучении, — одно и то же распределение. У AE между кодами и $N(0,I)$ нет никакой связи (его `|mean|` кодов ≈ 2.46), и семпл из приора уходит в область, где декодер ничего осмысленного не выучил.

❓ **Inline Question 4.** В какой новой ситуации ты потянешься именно за VAE (а не за обычным AE или, скажем, за retrieval по эмбеддингам)? Сформулируй в терминах «нужно семплировать из латента» / «нужна плотность на латенте».

## Сравнение со статьёй

В работе основной график — оценка нижней границы (ELBO) и качество сэмплов на MNIST/Frey Face; мы взяли крошечный `load_digits`, но воспроизвели *качественный* эффект: обученный через ELBO декодер генерирует из приора когерентные объекты, а автоэнкодер той же ёмкости — нет. Абсолютные числа BCE/расстояний не сопоставимы (другой датасет, разрешение), сопоставим знак и величина контраста VAE↔AE.

> **Совпало ли с твоими ⛏ гипотезами из начала?** …

## Рефлексия

**Что забрать в работу:**
- reparameterization как общий приём: любой раз, когда под матожиданием стоят параметры распределения, из которого семплишь, — вынеси стохастику в независимый шум, чтобы потёк градиент.
- честный генеративный тест — не train-loss, а «семпл из приора → метрика на реальном многообразии», с тривиальными полами (data-mean обманчиво силён).

**Когда VAE — правильный инструмент:** …

**Что осталось непонятным → следующий узел beam:**
- posterior collapse (KL душит информативность кода) — когда и почему;
- $\beta$-VAE и цена/выгода перекоса баланса термов;
- непрерывный декодер (Gaussian вместо Bernoulli) — что меняется в reconstruction-терме.

> …

## Bonus — «Going deeper» (без баллов)

1. **KL-annealing / β-VAE:** домножь KL на $\beta$ и просвипируй $\beta\in\{0, 0.1, 1, 4\}$. Как двигаются `d_vae`, `code-space |mean|` и reconstruction BCE? При $\beta=0$ VAE вырождается в AE — проверь, что генеративный эффект пропадает.
2. **Posterior collapse:** увеличь `Z_DIM` до 64 и/или урежь декодер. Сколько латентных осей реально используются (по `mu.std(0)`)?
3. **Непрерывный декодер:** замени Bernoulli-NLL на гауссов reconstruction (MSE + константа). Меняется ли эффект?

> Лог эксперимента: …